In [ ]:
# ================================
# 🚀 ConvNeXt - Image Classification
# ================================
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ConvNeXtTiny
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
import os

# ================================
# 🔧 Paths (à modifier selon ton dataset)
# ================================
train_dir = "/content/drive/MyDrive/dataset_product/train"
val_dir = "/content/drive/MyDrive/dataset_product/val"
test_dir = "/content/drive/MyDrive/dataset_product/test"

# ================================
# ⚙ Préparation du dataset
# ================================
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

num_classes = len(train_generator.class_indices)
print("Nombre de classes :", num_classes)

# ================================
# 🧠 Construction du modèle ConvNeXt
# ================================
base_model = ConvNeXtTiny(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # On gèle les poids pré-entraînés au début

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.4)(x)
output = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

# ================================
# 🧩 Compilation
# ================================
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ================================
# 🏋 Entraînement
# ================================
epochs = 20

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs
)

# ================================
# 🔓 Débloquer le base model pour fine-tuning
# ================================
base_model.trainable = True
model.compile(
    optimizer=Adam(learning_rate=1e-5),  # learning rate plus bas pour fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

fine_tune_epochs = 10
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=fine_tune_epochs
)

# ================================
# 🧪 Évaluation finale
# ================================
loss, acc = model.evaluate(test_generator)
print(f"✅ Test Accuracy : {acc*100:.2f}%")

# ================================
# 💾 Sauvegarde du modèle
# ================================
model.save("convnext_model.keras")
print("Modèle sauvegardé sous 'convnext_model.keras'")

Found 2807 images belonging to 7 classes.
Found 356 images belonging to 7 classes.
Found 280 images belonging to 7 classes.
Nombre de classes : 7
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_pres… │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stem  │ (None, 56, 56,    │      4,896 │ convnext_tiny_pr… │
│ (Sequential)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │      4,800 │ convnext_tiny_st… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │        192 │ convnext_tiny_st… │
│ (LayerNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │     37,248 │ convnext_tiny_st… │
│ (Dense)             │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │          0 │ convnext_tiny_st… │
│ (Activation)        │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │     36,960 │ convnext_tiny_st… │
│ (Dense)             │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │         96 │ convnext_tiny_st… │
│ (LayerScale)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │          0 │ convnext_tiny_st… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 56, 56,    │          0 │ convnext_tiny_st… │
│                     │ 96)               │            │ convnext_tiny_st… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │      4,800 │ add[0][0]         │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │        192 │ convnext_tiny_st… │
│ (LayerNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │     37,248 │ convnext_tiny_st… │
│ (Dense)             │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │          0 │ convnext_tiny_st… │
│ (Activation)        │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │     36,960 │ convnext_tiny_st… │
│ (Dense)             │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_tiny_stag… │ (None, 56, 56,    │         96 │ convnext_tiny_st

 Total params: 27,825,511 (106.15 MB)

 Trainable params: 5,383 (21.03 KB)

 Non-trainable params: 27,820,128 (106.13 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 2827s 32s/step - accuracy: 0.1382 - loss: 2.8047 - val_accuracy: 0.1545 - val_loss: 2.1211
Epoch 2/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 45s 507ms/step - accuracy: 0.1366 - loss: 2.4329 - val_accuracy: 0.2219 - val_loss: 1.9456
Epoch 3/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 46s 519ms/step - accuracy: 0.1495 - loss: 2.2039 - val_accuracy: 0.3427 - val_loss: 1.8611
Epoch 4/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 44s 500ms/step - accuracy: 0.1819 - loss: 2.1101 - val_accuracy: 0.3848 - val_loss: 1.8105
Epoch 5/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 44s 503ms/step - accuracy: 0.1838 - loss: 2.0790 - val_accuracy: 0.4803 - val_loss: 1.7638
Epoch 6/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 45s 505ms/step - accuracy: 0.2241 - loss: 1.9729 - val_accuracy: 0.6124 - val_loss: 1.7181
Epoch 7/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 46s 518ms/step - accuracy: 0.2503 - loss: 1.9006 - val_accuracy: 0.6236 - val_loss: 1.6783
Epoch 8/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 45s 511ms/step - accuracy: 0.2838 - loss: 1.8553 - val_accu